# EDL Full Experiment (auto-parallel)

10-fold cross-validation comparison of 8 LDL models across 3 datasets
(`SJAFFE`, `SBU_3DFE`, `Human_Gene`).

The pool config is **auto-detected** by `edl_workers.auto_pool_config()`:

- **GPU mode** if ≥ 2 CUDA GPUs are visible to `nvidia-smi` — one worker
  pinned per GPU.
- **CPU mode** otherwise — a small number of fat CPU workers (≈
  `cpu_count // 4`) so TF's intra/inter-op threads don't oversubscribe.

The actual training loop lives in `edl_workers.py` next to this notebook
so loky subprocesses can `import edl_workers` cleanly. TensorFlow is
imported lazily inside each worker after `CUDA_VISIBLE_DEVICES` is pinned.

For each (model, dataset) pair we record six distributional metrics
(`chebyshev`, `clark`, `canberra`, `kl_divergence`, `cosine`, `intersection`)
across 10 folds with a 10% test split per fold, then summarise as
mean ± std.

For the evidential models (`EDL_LDL`, `BEDL_LDL`) and `SNEFY_LDL` we also
report:

- **Mean uncertainty** — average per-sample uncertainty on the test set
  (lower = the model is more confident).
- **Uncertainty calibration (Spearman ρ)** — rank correlation between
  per-sample uncertainty and per-sample KL divergence error.
  Higher = uncertainty tracks error better.


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
import multiprocessing as mp
from collections import defaultdict
from concurrent.futures import as_completed
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from loky import get_reusable_executor

# Make edl_workers importable whether the kernel started in `demo/` or in the
# project root.
_demo_dir = Path.cwd() if (Path.cwd() / 'edl_workers.py').exists() else Path.cwd() / 'demo'
if str(_demo_dir) not in sys.path:
    sys.path.insert(0, str(_demo_dir))

from pyldl.utils import load_dataset
from edl_workers import (
    init_worker, run_one_fold, auto_pool_config,
    MODEL_NAMES, METRICS,
)


## Configuration

`auto_pool_config()` picks GPU vs CPU mode based on what's actually present.
Override `MODE_OVERRIDE` if you want to force one or the other (useful for
A/B testing GPU-vs-CPU on the same box).


In [ ]:
DATASETS = ['SJAFFE', 'SBU_3DFE', 'Human_Gene']
N_SPLITS = 10
N_EPOCHS = 100
RANDOM_STATE = 0

# --- Parallel config (auto-detected) -------------------------------------
# Set to None to auto-detect, or pass a dict to override e.g.
#   POOL_CFG = {'mode': 'CPU', 'gpu_ids': [], 'n_workers': 4,
#               'intra_op_threads': 1, 'inter_op_threads': 1}
POOL_CFG = None
# -------------------------------------------------------------------------

if POOL_CFG is None:
    POOL_CFG = auto_pool_config()

GPU_IDS   = POOL_CFG['gpu_ids']
N_WORKERS = POOL_CFG['n_workers']
INTRA     = POOL_CFG['intra_op_threads']
INTER     = POOL_CFG['inter_op_threads']
MODE      = POOL_CFG['mode']

print(f'mode      : {MODE}')
print(f'workers   : {N_WORKERS}')
print(f'gpu_ids   : {GPU_IDS if GPU_IDS else "(CPU only)"}')
print(f'tf threads: intra={INTRA}, inter={INTER}')


## Build the worker pool

Each worker pulls one entry off `gpu_queue` exactly once at startup
(loky's `initializer`) and pins `CUDA_VISIBLE_DEVICES` before TF sees a
GPU. In CPU mode the entry is `None`, which triggers
`tf.config.set_visible_devices([], 'GPU')` so TF can't accidentally find
the GPU through another path. `reuse=False` forces a fresh pool if you
re-run this cell after changing the config.


In [ ]:
mgr = mp.Manager()
gpu_queue = mgr.Queue()

slots = list(GPU_IDS) if GPU_IDS else [None] * N_WORKERS
assert len(slots) == N_WORKERS, 'one queue slot per worker'
for g in slots:
    gpu_queue.put(g)

executor = get_reusable_executor(
    max_workers=N_WORKERS,
    initializer=init_worker,
    initargs=(gpu_queue, INTRA, INTER),
    reuse=False,
)
print(f'pool ready: {N_WORKERS} workers, slots={slots}')


## Sanity check: what do the workers actually see?

Submit one cheap probe per worker and confirm that:

- **GPU mode**: each worker reports a different `CUDA_VISIBLE_DEVICES`
  and a single GPU device.
- **CPU mode**: every worker reports `CUDA_VISIBLE_DEVICES = '-1'` and
  an empty `gpu_devices` list.

If a row shows the wrong state, the pool's pinning didn't take effect —
re-run the pool cell to get a fresh pool.


In [ ]:
def _check():
    import os, tensorflow as tf
    return {
        'pid': os.getpid(),
        'CUDA_VISIBLE_DEVICES': os.environ.get('CUDA_VISIBLE_DEVICES'),
        'gpu_devices': [d.name for d in tf.config.list_physical_devices('GPU')],
        'intra_threads': tf.config.threading.get_intra_op_parallelism_threads(),
        'inter_threads': tf.config.threading.get_inter_op_parallelism_threads(),
    }

# Submit several so loky has a reason to spin up every worker.
checks = [executor.submit(_check) for _ in range(N_WORKERS * 4)]
df = pd.DataFrame([c.result() for c in checks]).drop_duplicates(subset='pid').reset_index(drop=True)
print(f'{len(df)} unique workers (expected {N_WORKERS})')
df


## Build the job list

`KFold` splits are generated in the parent (deterministic, cheap) and the
fold slices are passed to workers as numpy arrays. Loky memmaps large
numpy arrays automatically, so this is fast even for the bigger datasets.


In [ ]:
jobs = []
for dataset_name in DATASETS:
    X, D = load_dataset(dataset_name, dir='dataset')
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        Xtr, Xte = X[train_idx], X[test_idx]
        Dtr, Dte = D[train_idx], D[test_idx]
        for model_name in MODEL_NAMES:
            jobs.append((dataset_name, model_name, fold_idx, Xtr, Dtr, Xte, Dte))

total = len(jobs)
print(f'queued {total} jobs ({len(DATASETS)} datasets × {N_SPLITS} folds × {len(MODEL_NAMES)} models)')


## Submit + collect

Submission is non-blocking; results stream back via `as_completed` so the
log shows progress as folds finish. Per-fold failures are caught and
printed but don't stop the run.


In [ ]:
futures = {
    executor.submit(run_one_fold, ds, m, fi, Xtr, Dtr, Xte, Dte, N_EPOCHS): (ds, m, fi)
    for (ds, m, fi, Xtr, Dtr, Xte, Dte) in jobs
}

raw_results = []
failures = []
for i, fut in enumerate(as_completed(futures), start=1):
    ds, m, fi = futures[fut]
    try:
        raw_results.append(fut.result())
        status = 'ok'
    except Exception as e:
        failures.append((ds, m, fi, repr(e)))
        status = f'FAILED ({type(e).__name__}: {e})'
    print(f'[{i:4d}/{total}] {ds:12s} | fold {fi:2d} | {m:30s} {status}')

print(f'\ndone: {len(raw_results)} ok, {len(failures)} failed')


## Bucket results into per-model DataFrames

`per_model_results[(dataset, model_name)]` is a DataFrame with one row per
fold; columns are the recorded metrics.


In [ ]:
buckets = defaultdict(list)
for r in raw_results:
    buckets[(r['dataset'], r['model'])].append(r['scores'])

per_model_results = {key: pd.DataFrame(rows) for key, rows in buckets.items()}
print(f'{len(per_model_results)} (dataset, model) combinations have results')


## Per-model fold tables

Inspect any single (dataset, model) DataFrame:


In [ ]:
# Pick the first (dataset, model) that actually has results so this cell
# doesn't KeyError if you ran with a reduced DATASETS list.
if per_model_results:
    first_key = next(iter(per_model_results))
    print(f'showing: {first_key}')
    display(per_model_results[first_key])
else:
    print('no results yet — run the submit cell first')


## Combined summary — mean ± std across folds

One row per (dataset, model); columns are `metric_mean` / `metric_std`.


In [ ]:
def summarize(df):
    out = {}
    for col in df.columns:
        out[f'{col}_mean'] = df[col].mean()
        out[f'{col}_std']  = df[col].std()
    return out


summary_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name, **summarize(df)}
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index(['dataset', 'model'])
summary


### Compact view: `mean ± std` per metric


In [ ]:
def fmt(mean, std):
    if pd.isna(mean):
        return ''
    return f'{mean:.4f} ± {std:.4f}'


compact_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if df.empty:
        continue
    row = {'dataset': dataset_name, 'model': model_name}
    for col in df.columns:
        row[col] = fmt(df[col].mean(), df[col].std())
    compact_rows.append(row)

compact = pd.DataFrame(compact_rows).set_index(['dataset', 'model'])
compact


## Uncertainty results (EDL_LDL, BEDL_LDL, SNEFY_LDL only)

- `mean_uncertainty` — average per-sample uncertainty on test (model-specific
  scale; lower = more confident).
- `uncertainty_calibration` — Spearman ρ between per-sample uncertainty and
  per-sample KL divergence error. Higher = uncertainty better predicts error.


In [ ]:
uncertainty_models = {
    'EDL_LDL (loglikelihood)', 'EDL_LDL (bayes_mse)',
    'BEDL_LDL (loglikelihood)', 'BEDL_LDL (bayes_mse)',
    'SNEFY_LDL',
}

uncertainty_rows = []
for (dataset_name, model_name), df in per_model_results.items():
    if model_name not in uncertainty_models or df.empty:
        continue
    if 'mean_uncertainty' not in df.columns:
        continue
    uncertainty_rows.append({
        'dataset': dataset_name,
        'model': model_name,
        'mean_uncertainty':        fmt(df['mean_uncertainty'].mean(),        df['mean_uncertainty'].std()),
        'uncertainty_calibration': fmt(df['uncertainty_calibration'].mean(), df['uncertainty_calibration'].std()),
    })

uncertainty_summary = pd.DataFrame(uncertainty_rows).set_index(['dataset', 'model'])
uncertainty_summary


## Shut down the pool

Loky reuses pools by default; close it explicitly when you're done so the
worker processes (and any GPU memory they hold) are released.


In [ ]:
executor.shutdown(wait=True, kill_workers=True)
